In [2]:
#task 4 python code
import sqlite3
import pandas as pd


def create_database(db_name="lusitania_many_to_many.db"):
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON;")

    conn.execute("""
    CREATE TABLE IF NOT EXISTS passenger (
        passenger_id INTEGER PRIMARY KEY,
        family_name TEXT NOT NULL,
        personal_name TEXT NOT NULL,
        age REAL,
        sex TEXT
    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS cabin_inventory (
        cabin_id INTEGER PRIMARY KEY,
        cabin_class TEXT NOT NULL,
        no_of_beds INTEGER NOT NULL,
        price REAL,
        ship_level TEXT
    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS cabin_assignment (
        assignment_id INTEGER PRIMARY KEY AUTOINCREMENT,
        passenger_id INTEGER NOT NULL,
        cabin_id INTEGER NOT NULL,
        assigned_date TEXT DEFAULT CURRENT_DATE,
        FOREIGN KEY (passenger_id)
            REFERENCES passenger(passenger_id)
            ON DELETE CASCADE,
        FOREIGN KEY (cabin_id)
            REFERENCES cabin_inventory(cabin_id)
            ON DELETE CASCADE
    )
    """)

    conn.commit()
    return conn


def insert_sample_data(conn):
    passengers = [
        (1, "Smith", "John", 34, "M"),
        (2, "Smith", "Mary", 30, "F"),
        (3, "Brown", "William", 12, "M"),
        (4, "Jones", "Elizabeth", 28, "F"),
        (5, "Taylor", "James", 41, "M")
    ]

    cabins = [
        (101, "First", 4, 120.0, "A Deck"),
        (102, "Second", 3, 85.0, "B Deck"),
        (103, "Third", 6, 45.0, "C Deck")
    ]

    assignments = [
        (1, 101),
        (2, 101),
        (3, 102),
        (4, 103),
        (5, 103)
    ]

    conn.executemany("""
    INSERT OR REPLACE INTO passenger
    (passenger_id, family_name, personal_name, age, sex)
    VALUES (?, ?, ?, ?, ?)
    """, passengers)

    conn.executemany("""
    INSERT OR REPLACE INTO cabin_inventory
    (cabin_id, cabin_class, no_of_beds, price, ship_level)
    VALUES (?, ?, ?, ?, ?)
    """, cabins)

    conn.execute("DELETE FROM cabin_assignment")

    conn.executemany("""
    INSERT INTO cabin_assignment (passenger_id, cabin_id)
    VALUES (?, ?)
    """, assignments)

    conn.commit()


def show_assignments(conn, title="Current cabin assignments"):
    query = """
    SELECT
        a.assignment_id,
        p.passenger_id,
        p.personal_name,
        p.family_name,
        c.cabin_id,
        c.cabin_class,
        c.ship_level
    FROM cabin_assignment a
    JOIN passenger p
        ON a.passenger_id = p.passenger_id
    JOIN cabin_inventory c
        ON a.cabin_id = c.cabin_id
    ORDER BY c.cabin_id, p.family_name, p.personal_name
    """
    df = pd.read_sql(query, conn)
    print(f"\n{title}")
    print(df)


def move_passenger(conn, passenger_id, new_cabin_id):
    conn.execute("""
    UPDATE cabin_assignment
    SET cabin_id = ?
    WHERE passenger_id = ?
    """, (new_cabin_id, passenger_id))
    conn.commit()


def delete_passenger(conn, passenger_id):
    conn.execute("""
    DELETE FROM passenger
    WHERE passenger_id = ?
    """, (passenger_id,))
    conn.commit()


def delete_cabin(conn, cabin_id):
    conn.execute("""
    DELETE FROM cabin_inventory
    WHERE cabin_id = ?
    """, (cabin_id,))
    conn.commit()


def main():
    conn = create_database()
    insert_sample_data(conn)

    show_assignments(conn, "Before moving passenger")

    move_passenger(conn, passenger_id=3, new_cabin_id=103)
    show_assignments(conn, "After moving passenger 3 to cabin 103")

    delete_passenger(conn, 2)
    show_assignments(conn, "After deleting passenger 2 (cascade removes assignment)")

    delete_cabin(conn, 101)
    show_assignments(conn, "After deleting cabin 101 (cascade removes related assignments)")

    conn.close()


if __name__ == "__main__":
    main()


Before moving passenger
   assignment_id  passenger_id personal_name family_name  cabin_id  \
0              1             1          John       Smith       101   
1              2             2          Mary       Smith       101   
2              3             3       William       Brown       102   
3              4             4     Elizabeth       Jones       103   
4              5             5         James      Taylor       103   

  cabin_class ship_level  
0       First     A Deck  
1       First     A Deck  
2      Second     B Deck  
3       Third     C Deck  
4       Third     C Deck  

After moving passenger 3 to cabin 103
   assignment_id  passenger_id personal_name family_name  cabin_id  \
0              1             1          John       Smith       101   
1              2             2          Mary       Smith       101   
2              3             3       William       Brown       103   
3              4             4     Elizabeth       Jones       103   
4  